# Projeto: Execução da pipeline de dados utilizando notebooks workflow

### Objetivo  
Este notebook tem como objetivo realizar todo o processo de execução das camadas de dados, desde o consumo dos dados por meio de webscrapping e api's, até a execução de criação e persistência dos dados na camada gold. Finalizando com emissão de alertas.

## Componentes Importados

### Bibliotecas Externas:

- **concurent** utilizada função ThreadPoolExecutor para realizar execução de notebooks em paralelo.

---

## Histórico de Atualizações

| Data       | Responsável    | Versão | Descrição                                                                 |
|------------|----------------|--------|---------------------------------------------------------------------------|
| 19/05/2025 | Lucas Berta    | 1.0    | Criação do notebook workflow       |


In [0]:
%run ./workflow_utils

In [0]:
with ThreadPoolExecutor(max_workers=len(list(PIPELINES.values()))) as executor:
    results = executor.map(run_notebook, list(PIPELINES.values()))

In [0]:
run_notebook(NOTEBOOKS_GOLD["gold"])

In [0]:
gold_alertas = [NOTEBOOKS_GOLD["alerta_alagamento"], NOTEBOOKS_GOLD["clima_alagamento"], NOTEBOOKS_GOLD["previsao_clima"]]

with ThreadPoolExecutor(max_workers=len(gold_alertas)) as executor:
    results = executor.map(run_notebook, gold_alertas)

In [0]:
run_notebook(NEWSLETTER["newsletter"])

In [0]:
df = ler_tabelas("grupo01.gold.alertas_alagamentos")

In [0]:
df = df.filter(col("dt_carga") >= expr("current_timestamp() - INTERVAL 20 MINUTE"))

In [0]:
if df.count() > 0:
    headers = {"Content-Type": "application/json"}
    response = requests.post(TEAMS_WEBHOOK_URL, headers=headers, data=json.dumps(MESSAGE_CARD))
    if response.status_code == 200:
         print("Alerta enviado com sucesso para o Teams!")
    else:
         print("Erro ao enviar alerta para o Teams. Status Code:", response.status_code)
else:
    print("Nenhum alerta de alagamento foi detectado.")